# spaCy - Main Features Overview

This notebook walks through the core features of [spaCy](https://spacy.io/), an industrial-strength NLP library for Python.

## Table of Contents
1. Installation & Setup
2. Tokenization
3. Part-of-Speech (POS) Tagging
4. Named Entity Recognition (NER)
5. Dependency Parsing
6. Lemmatization
7. Sentence Segmentation
8. Word Vectors & Similarity
9. Rule-Based Matching
10. Custom Pipeline Components

## 1. Installation & Setup

Install spaCy and download a language model before getting started.

In [ ]:
# Install spaCy (uncomment if needed)
# !pip install spacy

# Download the small English model
# !python -m spacy download en_core_web_sm

# For word vectors, use the medium or large model instead:
# !python -m spacy download en_core_web_md

import spacy

# Load the small English pipeline
nlp = spacy.load("en_core_web_sm")
print(f"spaCy version: {spacy.__version__}")
print(f"Pipeline components: {nlp.pipe_names}")

## 2. Tokenization

spaCy splits text into meaningful units (tokens) — words, punctuation, numbers, etc.

In [ ]:
text = "Apple is looking at buying U.K. startup for $1 billion. That's a lot of money!"
doc = nlp(text)

print("Tokens:")
print("-" * 50)
for token in doc:
    print(f"{token.text:<15} | is_alpha: {token.is_alpha:<6} | is_punct: {token.is_punct:<6} | is_stop: {token.is_stop}")

In [ ]:
# Token attributes
print(f"{'Token':<12} {'Idx':<5} {'Len':<5} {'Shape':<8} {'Like_num':<10} {'Is_currency':<12}")
print("=" * 55)
for token in doc:
    print(f"{token.text:<12} {token.idx:<5} {len(token):<5} {token.shape_:<8} {token.like_num:<10} {token.is_currency:<12}")

## 3. Part-of-Speech (POS) Tagging

Each token is assigned a grammatical part-of-speech tag (noun, verb, adjective, etc.).

In [ ]:
text = "The quick brown fox jumps over the lazy dog near a flowing river."
doc = nlp(text)

print(f"{'Token':<12} {'POS':<8} {'Fine-grained':<14} {'Explanation'}")
print("=" * 60)
for token in doc:
    print(f"{token.text:<12} {token.pos_:<8} {token.tag_:<14} {spacy.explain(token.tag_)}")

In [ ]:
# Filter tokens by POS
nouns = [token.text for token in doc if token.pos_ == "NOUN"]
verbs = [token.text for token in doc if token.pos_ == "VERB"]
adjectives = [token.text for token in doc if token.pos_ == "ADJ"]

print(f"Nouns:      {nouns}")
print(f"Verbs:      {verbs}")
print(f"Adjectives: {adjectives}")

## 4. Named Entity Recognition (NER)

spaCy identifies real-world entities like people, organizations, locations, dates, and monetary values.

In [ ]:
text = (
    "Elon Musk founded SpaceX in 2002. The company is headquartered in "
    "Hawthorne, California. NASA awarded SpaceX a $2.9 billion contract "
    "in April 2021 for the Artemis program."
)
doc = nlp(text)

print(f"{'Entity':<25} {'Label':<12} {'Description'}")
print("=" * 65)
for ent in doc.ents:
    print(f"{ent.text:<25} {ent.label_:<12} {spacy.explain(ent.label_)}")

In [ ]:
# Visualize entities inline (renders in Jupyter)
from spacy import displacy

displacy.render(doc, style="ent", jupyter=True)

## 5. Dependency Parsing

spaCy analyzes the grammatical structure of a sentence, determining how words relate to each other.

In [ ]:
text = "The cat sat on the mat and watched the birds."
doc = nlp(text)

print(f"{'Token':<10} {'Dep':<12} {'Head':<10} {'Children'}")
print("=" * 55)
for token in doc:
    children = [child.text for child in token.children]
    print(f"{token.text:<10} {token.dep_:<12} {token.head.text:<10} {children}")

In [ ]:
# Visualize the dependency tree
displacy.render(doc, style="dep", jupyter=True, options={"compact": True})

In [ ]:
# Extract noun chunks (base noun phrases)
print("Noun chunks:")
for chunk in doc.noun_chunks:
    print(f"  '{chunk.text}' (root: {chunk.root.text}, dep: {chunk.root.dep_})")

## 6. Lemmatization

Lemmatization reduces words to their base/dictionary form (e.g., "running" -> "run").

In [ ]:
text = "The striped bats were hanging on their feet and eating best fishes."
doc = nlp(text)

print(f"{'Token':<12} {'Lemma':<12} {'POS'}")
print("=" * 35)
for token in doc:
    if token.text != token.lemma_:
        print(f"{token.text:<12} {token.lemma_:<12} {token.pos_}  <-- changed")
    else:
        print(f"{token.text:<12} {token.lemma_:<12} {token.pos_}")

## 7. Sentence Segmentation

spaCy splits text into individual sentences.

In [ ]:
text = (
    "Natural language processing is a subfield of AI. "
    "It focuses on the interaction between computers and humans. "
    "Applications include translation, sentiment analysis, and chatbots. "
    "spaCy is one of the most popular NLP libraries!"
)
doc = nlp(text)

print(f"Found {len(list(doc.sents))} sentences:\n")
for i, sent in enumerate(doc.sents, 1):
    print(f"  {i}. {sent.text}")

## 8. Word Vectors & Similarity

spaCy models can compute semantic similarity between words, spans, and documents.

> **Note:** The small model (`en_core_web_sm`) has limited vectors. For meaningful similarity, use `en_core_web_md` or `en_core_web_lg`.

In [ ]:
# Document-level similarity
doc1 = nlp("I enjoy playing football on weekends.")
doc2 = nlp("Soccer is my favorite weekend sport.")
doc3 = nlp("The stock market crashed yesterday.")

print("Document Similarity Scores:")
print(f"  doc1 vs doc2 (related):   {doc1.similarity(doc2):.4f}")
print(f"  doc1 vs doc3 (unrelated): {doc1.similarity(doc3):.4f}")
print(f"  doc2 vs doc3 (unrelated): {doc2.similarity(doc3):.4f}")

In [ ]:
# Token-level similarity
doc = nlp("dog cat banana car")
tokens = list(doc)

print("Token Similarity Matrix:")
print(f"{'':>10}", end="")
for t in tokens:
    print(f"{t.text:>10}", end="")
print()

for t1 in tokens:
    print(f"{t1.text:>10}", end="")
    for t2 in tokens:
        print(f"{t1.similarity(t2):>10.4f}", end="")
    print()

## 9. Rule-Based Matching

spaCy's `Matcher` lets you find sequences of tokens based on patterns (like regex, but for linguistic features).

In [ ]:
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)

# Pattern: adjective followed by one or more nouns (e.g., "quick fox", "lazy dog")
pattern = [{"POS": "ADJ"}, {"POS": "NOUN", "OP": "+"}]
matcher.add("ADJ_NOUN", [pattern])

doc = nlp("The quick brown fox jumps over the lazy dog near the tall green tree.")
matches = matcher(doc)

print("Matches for pattern [ADJ + NOUN+]:")
for match_id, start, end in matches:
    span = doc[start:end]
    print(f"  '{span.text}' (tokens {start}-{end})")

In [ ]:
from spacy.matcher import PhraseMatcher

# PhraseMatcher: efficient matching on exact phrases
phrase_matcher = PhraseMatcher(nlp.vocab)

technologies = ["machine learning", "deep learning", "natural language processing", "computer vision"]
patterns = [nlp.make_doc(tech) for tech in technologies]
phrase_matcher.add("TECH_TERMS", patterns)

doc = nlp(
    "The course covers machine learning and deep learning. "
    "Students also study natural language processing and computer vision."
)
matches = phrase_matcher(doc)

print("Technology terms found:")
for match_id, start, end in matches:
    print(f"  '{doc[start:end].text}'")

## 10. Custom Pipeline Components

You can add your own processing steps to the spaCy pipeline using the `@Language.component` decorator.

In [ ]:
from spacy.language import Language


@Language.component("word_counter")
def word_counter(doc):
    """Custom component that counts words (non-punctuation, non-space tokens)."""
    word_count = sum(1 for token in doc if not token.is_punct and not token.is_space)
    doc._.word_count = word_count
    return doc


# Register a custom extension attribute on Doc
from spacy.tokens import Doc

if not Doc.has_extension("word_count"):
    Doc.set_extension("word_count", default=0)

# Add to pipeline
if "word_counter" not in nlp.pipe_names:
    nlp.add_pipe("word_counter", last=True)

print(f"Updated pipeline: {nlp.pipe_names}")

doc = nlp("spaCy is an amazing library for natural language processing!")
print(f"Text: '{doc.text}'")
print(f"Word count: {doc._.word_count}")

## Summary

| Feature | Description | Key API |
|---|---|---|
| **Tokenization** | Split text into tokens | `nlp(text)`, `token.text` |
| **POS Tagging** | Grammatical categories | `token.pos_`, `token.tag_` |
| **NER** | Real-world entity detection | `doc.ents`, `ent.label_` |
| **Dependency Parsing** | Syntactic relationships | `token.dep_`, `token.head` |
| **Lemmatization** | Base word forms | `token.lemma_` |
| **Sentence Segmentation** | Split into sentences | `doc.sents` |
| **Similarity** | Semantic comparison | `doc.similarity()` |
| **Matching** | Pattern-based token search | `Matcher`, `PhraseMatcher` |
| **Custom Components** | Extend the pipeline | `@Language.component` |

### Further Resources
- [spaCy Documentation](https://spacy.io/usage)
- [spaCy 101](https://spacy.io/usage/spacy-101)
- [Available Models](https://spacy.io/models)
- [API Reference](https://spacy.io/api)